# 🚀 WoW Bot Training mit YOLO8 auf Google Colab

Dieses Notebook trainiert ein YOLO8-Modell für den WoW Bot mit GPU-Beschleunigung.

## 📋 Voraussetzungen
- Google Colab Account (kostenlos)
- Roboflow API Key
- Trainingsdaten auf Roboflow hochgeladen


## 1️⃣ Setup & Installation


In [ ]:
# GPU aktivieren: Runtime -> Change runtime type -> Hardware accelerator: GPU
# PyTorch neu installieren (behebt Import-Probleme und zirkuläre Abhängigkeiten)
import subprocess
import sys

# Lösche eventuell vorhandene PyTorch-Installationen
print("🧹 Bereinige alte PyTorch-Installation...")
try:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"], 
                   capture_output=True, check=False)
except:
    pass

# Installiere PyTorch mit CUDA-Unterstützung (für Colab GPUs)
print("🔄 Installiere PyTorch mit CUDA-Unterstützung...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "torchvision", "torchaudio", 
                       "--index-url", "https://download.pytorch.org/whl/cu121", "-q"])
print("✅ PyTorch Installation abgeschlossen\n")

# Prüfe ob GPU aktiviert ist
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA verfügbar: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("⚠️ WARNUNG: Keine GPU erkannt! Bitte GPU in Runtime-Einstellungen aktivieren.")


PyTorch Version: 2.9.0+cpu
CUDA verfügbar: False
⚠️ WARNUNG: Keine GPU erkannt! Bitte GPU in Runtime-Einstellungen aktivieren.


In [2]:
# Installiere alle benötigten Pakete
%pip install ultralytics roboflow -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 107.9 MB/s eta 0:00:0000:01


## 2️⃣ Konfiguration

**WICHTIG:** Trage hier deine Roboflow-Daten ein!


In [ ]:
# ============================================
# KONFIGURATION - BITTE ANPASSEN!
# ============================================

# Dein Roboflow API Key (findest du auf roboflow.com unter Account Settings)
ROBOFLOW_API_KEY = "bWCQrHFXnnZsy0GLxM8c"

# Roboflow Projekt-Daten (aus deinem Roboflow-Projekt)
WORKSPACE = "icq"
PROJECT = "skull-fupdu"
VERSION = 5

# Training-Parameter
EPOCHS = 200          # Anzahl der Training-Durchläufe
IMG_SIZE = 640       # Bildgröße (640, 416 oder 320 - kleiner = weniger GPU-Speicher)
BATCH_SIZE = 16      # Batch-Größe (16 ist Standard, bei Out-of-Memory auf 8, 4 oder 2 reduzieren)

print("✅ Konfiguration geladen!")


✅ Konfiguration geladen!


## 3️⃣ Daten von Roboflow laden


In [ ]:
from roboflow import Roboflow

# Verbinde mit Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
version = project.version(VERSION)

# Lade Datensatz im YOLOv8-Format
print("📦 Lade Trainingsdaten von Roboflow...")
dataset = version.download("yolov8")
print(f"✅ Datensatz geladen: {dataset.location}")


loading Roboflow workspace...
loading Roboflow project...
📦 Lade Trainingsdaten von Roboflow...
Exporting format yolov8 in progress : 85.0%
Version export complete for yolov8 format



Extracting Dataset Version Zip to Skull-5 in yolov8:: 100%|██████████| 970/970 [00:00<00:00, 7003.27it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Datensatz geladen: /content/Skull-5


## 4️⃣ Training starten

Das Training kann je nach Datensatz-Größe und Epochs mehrere Stunden dauern. Colab bietet kostenlose GPU-Stunden, aber die Session kann nach ~12 Stunden Inaktivität beendet werden.


In [ ]:
from ultralytics import YOLO
import torch

# GPU-Erkennung
if torch.cuda.is_available():
    DEVICE = 0  # Erste GPU verwenden
    print(f"✓ GPU erkannt: {torch.cuda.get_device_name(0)}")
    print(f"✓ CUDA Version: {torch.version.cuda}")
else:
    DEVICE = "cpu"
    print("⚠ Keine GPU gefunden - Training läuft auf CPU (sehr langsam!)")
    print("💡 Tipp: Runtime -> Change runtime type -> Hardware accelerator: GPU")

# Lade YOLO8 Nano-Modell (kleinste Version für schnelle Inferenz)
print("\n📥 Lade YOLO8 Modell...")
model = YOLO("yolov8n.pt")

print("\n" + "=" * 60)
print("🚀 Training startet...")
print("=" * 60)
print(f"📦 Datensatz: {dataset.location}")
print(f"🎯 Epochs: 200")
print(f"⏱️  Patience: 50")
print(f"📏 Bildgröße: 640")
print(f"📊 Batch-Größe: 16")
print(f"🔧 Optimizer: auto")
print(f"💾 Save: True")
print(f"💻 Device: {DEVICE}")
print("\n")

# Starte Training
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=200,        # Höheres Limit
    patience=50,       # Etwas mehr Geduld, da der Datensatz größer ist
    batch=16,          # Stabilisiert den Lernprozess
    imgsz=640,         # Standard für das Training
    optimizer='auto',  # Überlässt YOLO die Wahl des besten Lernalgorithmus
    save=True,         # Speichert Modell-Checkpoints
    plots=True,        # Erstellt schöne Diagramme
    device=DEVICE,     # GPU oder CPU
    amp=True,          # Mixed Precision Training (spart Speicher)
    workers=8,         # Mehr Worker für Colab
    project="runs",    # Speicherort
    name="detect/train" # Unterordner
)

print("\n" + "=" * 60)
print("✅ Training abgeschlossen!")
print("=" * 60)


⚠ Keine GPU gefunden - Training läuft auf CPU (sehr langsam!)
💡 Tipp: Runtime -> Change runtime type -> Hardware accelerator: GPU

📥 Lade YOLO8 Modell...

🚀 Training startet...
📦 Datensatz: /content/Skull-5
🎯 Epochs: 200
⏱️  Patience: 50
📏 Bildgröße: 640
📊 Batch-Größe: 16
🔧 Optimizer: auto
💾 Save: True
💻 Device: cpu


Ultralytics 8.3.241 🚀 Python-3.12.12 torch-2.9.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Skull-5/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7,

      1/200         0G      2.383       10.4      1.285         38        640: 85% ━━━━━━━━━━── 23/27 12.0s/it 4:52<47.9s


KeyboardInterrupt: 

## 5️⃣ Ergebnisse anzeigen


In [ ]:
# Zeige Trainings-Ergebnisse
import os

best_model_path = "runs/detect/train/weights/best.pt"
last_model_path = "runs/detect/train/weights/last.pt"

if os.path.exists(best_model_path):
    file_size = os.path.getsize(best_model_path) / (1024 * 1024)  # MB
    print(f"✅ Bestes Modell: {best_model_path}")
    print(f"   Größe: {file_size:.2f} MB")
else:
    print(f"❌ Modell nicht gefunden: {best_model_path}")

if os.path.exists(last_model_path):
    file_size = os.path.getsize(last_model_path) / (1024 * 1024)  # MB
    print(f"✅ Letztes Modell: {last_model_path}")
    print(f"   Größe: {file_size:.2f} MB")

# Zeige Trainings-Diagramme
from IPython.display import Image, display

results_dir = "runs/detect/train"
if os.path.exists(f"{results_dir}/results.png"):
    print("\n📊 Trainings-Diagramme:")
    display(Image(f"{results_dir}/results.png"))
if os.path.exists(f"{results_dir}/confusion_matrix.png"):
    display(Image(f"{results_dir}/confusion_matrix.png"))


## 6️⃣ Modell herunterladen

Nach dem Training kannst du das trainierte Modell auf deinen Computer herunterladen.


In [ ]:
# Erstelle ZIP-Archiv mit dem trainierten Modell
import zipfile
import os

best_model = "runs/detect/train/weights/best.pt"
zip_filename = "wow_bot_model.zip"

if os.path.exists(best_model):
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(best_model, "best.pt")
        # Füge auch data.yaml hinzu falls vorhanden
        data_yaml = f"{dataset.location}/data.yaml"
        if os.path.exists(data_yaml):
            zipf.write(data_yaml, "data.yaml")
    
    print(f"✅ ZIP-Archiv erstellt: {zip_filename}")
    print(f"📥 Klicke mit Rechtsklick auf die Datei im Datei-Explorer und wähle 'Download'")
    print(f"\n💡 Oder nutze die nächste Zelle zum direkten Download")
else:
    print(f"❌ Modell nicht gefunden: {best_model}")


In [ ]:
# Direkter Download (optional)
from google.colab import files

if os.path.exists("wow_bot_model.zip"):
    files.download("wow_bot_model.zip")
    print("✅ Download gestartet!")
else:
    print("❌ ZIP-Datei nicht gefunden. Bitte zuerst die Zelle darüber ausführen.")


## 🔧 Troubleshooting

### Out-of-Memory Fehler?
- Reduziere `BATCH_SIZE` auf 8, 4 oder 2
- Reduziere `IMG_SIZE` auf 416 oder 320

### Training zu langsam?
- Stelle sicher, dass GPU aktiviert ist (Runtime -> Change runtime type)
- Prüfe ob GPU erkannt wird in der ersten Zelle

### Session beendet?
- Colab beendet Sessions nach ~12 Stunden Inaktivität
- Speichere regelmäßig dein Modell (Zelle 6)
- Nutze Colab Pro für längere Sessions

### Modell nicht gefunden?
- Prüfe ob Training erfolgreich abgeschlossen wurde
- Schaue in den `runs/detect/train/` Ordner
